# 02 — Experiment Workbench

Use this notebook to create an ignored local config, run workflow steps, inspect training outputs, and append experiment-log rows. Long-running cells are gated by `RUN_LONG_STEPS`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

def find_repo_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "config" / "competition.yaml").is_file():
            return path
    raise FileNotFoundError("Could not find repo root containing config/competition.yaml")

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))
print(REPO_ROOT)

In [ ]:
from copy import deepcopy
from datetime import date
import yaml

from ua_detrac_starter.config import load_config

BASE_CONFIG = "config/competition.yaml"
LOCAL_CONFIG = "config/config.yaml"  # ignored by git

EXPERIMENT_ID = "exp001_baseline"
RUN_NAME = "exp001_baseline"
EPOCHS = 10
BATCH_SIZE = 16
LR0 = 0.01
IMAGE_SIZE = 640
PREDICT_CONF = 0.25
PREDICT_IOU = 0.70
NOTES = "baseline from latest 3LC revisions"
RUN_LONG_STEPS = False

base_cfg, _ = load_config(BASE_CONFIG)
exp_cfg = deepcopy(base_cfg)
exp_cfg.setdefault("training", {}).update({
    "run_name": RUN_NAME,
    "run_description": NOTES,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "image_size": IMAGE_SIZE,
    "lr0": LR0,
})
exp_cfg.setdefault("predict", {}).update({
    "conf": PREDICT_CONF,
    "iou": PREDICT_IOU,
})

local_config_path = REPO_ROOT / LOCAL_CONFIG
local_config_path.write_text(yaml.safe_dump(exp_cfg, sort_keys=False), encoding="utf-8")
print(f"Wrote {local_config_path}")
print(f"Use CONFIG={LOCAL_CONFIG}")

In [ ]:
def run_command(command, *, check=False):
    print("$", " ".join(command))
    result = subprocess.run(
        command,
        cwd=REPO_ROOT,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout)
    print("returncode=", result.returncode)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(command)}")
    return result

run_command(["make", "verify"])
run_command(["make", "verify-setup", f"CONFIG={LOCAL_CONFIG}"])

## Register, Train, Predict

Set `RUN_LONG_STEPS = True` in the config cell when you intentionally want to run registration/training/prediction from the notebook.

In [ ]:
if RUN_LONG_STEPS:
    run_command(["make", "register-tables", f"CONFIG={LOCAL_CONFIG}"], check=True)
    run_command(["make", "train", f"CONFIG={LOCAL_CONFIG}"], check=True)
    run_command(["make", "predict", f"CONFIG={LOCAL_CONFIG}"], check=True)
    run_command(["make", "validate-submission", "SAMPLE_SUBMISSION=data/competition_starter/sample_submission.csv", "SUBMISSION=submissions/submission.csv"], check=True)
else:
    print("RUN_LONG_STEPS is False. Commands are ready but not executed.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

results_csv = REPO_ROOT / "runs" / "detect" / RUN_NAME / "results.csv"
if results_csv.is_file():
    results = pd.read_csv(results_csv)
    display(results.tail())
    metric_cols = [col for col in results.columns if "metrics/" in col or "loss" in col]
    if metric_cols:
        results[metric_cols].plot(figsize=(12, 5), title=RUN_NAME)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
else:
    print(f"No results file yet: {results_csv}")

In [ ]:
PUBLIC_SCORE = ""
LOCAL_MAP50 = ""

log_command = [
    "python3", "scripts/make_experiment_entry.py",
    "--experiment-id", EXPERIMENT_ID,
    "--run-name", RUN_NAME,
    "--epochs", str(EPOCHS),
    "--batch", str(BATCH_SIZE),
    "--seed", str(exp_cfg.get("reproducibility", {}).get("seed", "")),
    "--notes", NOTES,
]

print("Experiment log command:")
print(" ".join(log_command))
# Uncomment when you want to append the row:
# run_command(log_command, check=True)

## Experiment Notes Template

- Hypothesis:
- Config change:
- 3LC revision used:
- Local metric:
- Kaggle public score:
- Error pattern improved:
- Error pattern worsened:
- Next action: